<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>
<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;"><h1 style="margin:0;"><b>Filtering in Frequency Domain — Algorithm</b></h1></div>


## Purpose and reading order

This notebook is the algorithm specification for the corresponding `main.ipynb` in this lab. It has two linked layers:

| Layer | Content | Question answered |
| --- | --- | --- |
| **Part I — Complete pseudocode** | One self-contained executable-style block with explicit branches and gates | What is the whole pipeline? |
| **Part II — Auditable specification** | Six-section contract for every step: Purpose, Inputs, Procedure, Produces, Acceptance, Failure mode | Why is each step valid? |

Names, shapes, thresholds, and failure decisions are kept identical between both layers. The acceptance contract of step *k* is the precondition of step *k+1*.

## Part I — Complete pseudocode

The single block below is the complete pipeline. Every named artifact is assigned before use; every failure is explicit. It is a direct, implementation-neutral transcription of the executable notebook.

```text
ALGORITHM Frequency_Domain_Filtering

# Algorithm 1 — Setup — Environment and Configuration
CALL ARRAY.set_printoptions(precision = 3, suppress = TRUE)
CALL EMIT("NumPy:", ARRAY.__version__)
CALL EMIT("Setup: PASS")

# Algorithm 2 — 0.1 Locate the Lab Automatically
PROCEDURE locate_lab_root()
    cwd ← PATH.cwd().resolve()
    FOR candidate IN [cwd, EXPAND cwd.parents] DO
        IF ((candidate / "data").is_dir() AND ((candidate / "notebooks") / "main.ipynb").is_file()) THEN
            RETURN candidate
        END IF
    END FOR
    RAISE FileNotFoundError("Could not locate the frequency-domain lab root.")
END PROCEDURE
LAB_ROOT ← locate_lab_root()
DATA_DIR ← (LAB_ROOT / "data")
OUTPUT_DIR ← ((LAB_ROOT / "outputs") / "figures")
CALL OUTPUT_DIR.mkdir(parents = TRUE, exist_ok = TRUE)
IMAGE_EXTENSIONS ← {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATASETS ← MAP{folder.name ↦ SORT(STREAM[path FOR path IN folder.rglob("*") WHERE (path.is_file() AND (path.suffix.lower() IN IMAGE_EXTENSIONS))]) FOR folder IN DATA_DIR.iterdir() WHERE folder.is_dir()}
FOR group IN ("Fourier", "PhaseMag", "Frequency") DO
    IF (NOT DATASETS.get(group)) THEN
        RAISE FileNotFoundError(FORMAT("No supported images found in data/{0}/" , group))
    END IF
END FOR
PROCEDURE select_input_image(group, VARIADIC keywords)
    keywords ← TUPLE(STREAM[keyword.lower() FOR keyword IN keywords])
    matches ← LIST[path FOR path IN DATASETS[group] WHERE ALL(STREAM[(keyword IN path.stem.lower()) FOR keyword IN keywords])]
    IF (NOT matches) THEN
        RAISE FileNotFoundError(FORMAT("No image matching {0} found in data/{1}/" , keywords, group))
    END IF
    RETURN matches[0]
END PROCEDURE
CALL EMIT("Lab root:", LAB_ROOT)
CALL EMIT("Data dir:", DATA_DIR)
FOR (group, files) IN ITEMS(DATASETS) DO
    CALL EMIT(FORMAT("{0:10s}: {1} image(s)" , group, LENGTH(files)))
END FOR
CALL EMIT("Output  :", OUTPUT_DIR)

# Algorithm 3 — 0.2 Reusable helpers
PROCEDURE load_grayscale_image(path)
    RETURN ARRAY.asarray(IMAGE.open(path).convert("L"), dtype = ARRAY.float32)
END PROCEDURE
PROCEDURE normalize_to_unit_interval(array)
    array ← ARRAY.asarray(array, dtype = ARRAY.float64)
    lo ← MIN(array)
    hi ← MAX(array)
    IF ARRAY.isclose(lo, hi) THEN
        RETURN ARRAY.zeros_like(array)
    END IF
    RETURN ((array − lo) / (hi − lo))
END PROCEDURE
PROCEDURE display_grayscale_image(ax, image, title, cmap = "gray")
    CALL ax.imshow(image, cmap = cmap)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END PROCEDURE
PROCEDURE save_output_figure(fig, filename)
    path ← (OUTPUT_DIR / filename)
    CALL fig.savefig(path, dpi = 160, bbox_inches = "tight")
    CALL EMIT("Saved:", path.name)
END PROCEDURE
PROCEDURE compute_centered_fft2(image)
    RETURN ARRAY.fft.fftshift(ARRAY.fft.fft2(image))
END PROCEDURE
PROCEDURE reconstruct_from_centered_fft2(centered_spectrum)
    RETURN ARRAY.real(ARRAY.fft.ifft2(ARRAY.fft.ifftshift(centered_spectrum)))
END PROCEDURE
PROCEDURE compute_log_spectrum_magnitude(centered_spectrum)
    RETURN ARRAY.log1p(ARRAY.abs(centered_spectrum))
END PROCEDURE
PROCEDURE compute_fft_roundtrip_tolerance(image, safety_factor = 32.0)
    image ← ARRAY.asarray(image)
    IF ARRAY.issubdtype(image.dtype, ARRAY.floating) THEN
        dtype ← image.dtype
    ELSE
        dtype ← ARRAY.float64
    END IF
    eps ← ARRAY.finfo(dtype).eps
    scale ← MAX(1.0, REAL(MAX(ARRAY, ARRAY.abs(image))))
    RETURN ((safety_factor * eps) * scale)
END PROCEDURE
PROCEDURE mean_squared_error(reference, test)
    reference ← ARRAY.asarray(reference, dtype = ARRAY.float64)
    test ← ARRAY.asarray(test, dtype = ARRAY.float64)
    RETURN MEAN(ARRAY, ((reference − test) ^ 2))
END PROCEDURE
PROCEDURE peak_signal_to_noise_ratio(reference, test, peak = 255.0)
    error ← mean_squared_error(reference, test)
    IF ARRAY.isclose(error, 0.0) THEN
        RETURN ARRAY.inf
    END IF
    RETURN (10 * ARRAY.log10(((peak ^ 2) / error)))
END PROCEDURE
PROCEDURE compute_mean_gradient_magnitude(image)
    (gy, gx) ← ARRAY.gradient(ARRAY.asarray(image, dtype = ARRAY.float64))
    RETURN MEAN(ARRAY, ARRAY.hypot(gx, gy))
END PROCEDURE
PROCEDURE measure_ringing_overshoot(image, low = 0.0, high = 255.0)
    image ← ARRAY.asarray(image, dtype = ARRAY.float64)
    above ← MAX(REAL((MAX(image) − high)), 0.0)
    below ← MAX(REAL((low − MIN(image))), 0.0)
    RETURN MAX(above, below)
END PROCEDURE

# Algorithm 4 — 1. Spatial-Frequency Characterization
width ← 512
x ← ARRAY.linspace(0, 1, width, endpoint = FALSE)
low_signal ← ARRAY.sin((((2 * ARRAY.pi) * 4) * x))
high_signal ← ARRAY.sin((((2 * ARRAY.pi) * 32) * x))
low_image ← ARRAY.tile(low_signal, (180, 1))
high_image ← ARRAY.tile(high_signal, (180, 1))
(fig, axes) ← PLOT.subplots(2, 2, figsize = (12, 6))
CALL axes[(0, 0)].plot(x, low_signal)
CALL axes[(0, 0)].set_title("4 cycles — low frequency")
CALL display_grayscale_image(axes[(0, 1)], low_image, "Low spatial frequency")
CALL axes[(1, 0)].plot(x, high_signal)
CALL axes[(1, 0)].set_title("32 cycles — high frequency")
CALL display_grayscale_image(axes[(1, 1)], high_image, "High spatial frequency")
CALL fig.tight_layout()
CALL save_output_figure(fig, "01_spatial_frequency.png")
CALL PLOT.show()

# Algorithm 5 — 2. 1-D DFT Validation with Synthetic Sinusoids
n ← ARRAY.arange(128)
signal ← (ARRAY.sin(((((2 * ARRAY.pi) * 5) * n) / LENGTH(n))) + (0.45 * ARRAY.sin(((((2 * ARRAY.pi) * 18) * n) / LENGTH(n)))))
signal_spectrum ← ARRAY.fft.fft(signal)
frequencies ← ARRAY.fft.fftfreq(LENGTH(signal))
(fig, axes) ← PLOT.subplots(1, 2, figsize = (12, 4))
CALL axes[0].plot(n, signal)
CALL axes[0].set_title("Signal = two sinusoids")
CALL axes[1].stem(frequencies[:64], ARRAY.abs(signal_spectrum[:64]))
CALL axes[1].set_title("FFT magnitude")
CALL axes[1].set_xlabel("Frequency")
CALL fig.tight_layout()
CALL save_output_figure(fig, "02_fft_1d.png")
CALL PLOT.show()

# Algorithm 6 — 3. The 2-D Fourier Transform for Images
HOUSE_PATH ← select_input_image("Fourier", "house")
house ← load_grayscale_image(HOUSE_PATH)
house_centered_spectrum ← compute_centered_fft2(house)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], house, FORMAT("{0} — spatial domain" , HOUSE_PATH.stem))
CALL display_grayscale_image(axes[1], compute_log_spectrum_magnitude(house_centered_spectrum),
    "Log magnitude spectrum")
CALL display_grayscale_image(axes[2], ARRAY.angle(house_centered_spectrum), "Phase spectrum",
    cmap = "twilight")
CALL fig.tight_layout()
CALL save_output_figure(fig, "03_house_spectrum.png")
CALL PLOT.show()

# Algorithm 7 — 4. 2-D Spectrum Interpretation
size ← 256
coords ← ARRAY.arange(size)
vertical ← ARRAY.tile(ARRAY.sin(((((2 * ARRAY.pi) * 16) * coords) / size)), (size, 1))
horizontal ← TRANSPOSE(vertical)
(xx, yy) ← ARRAY.meshgrid(coords, coords)
diagonal ← ARRAY.sin(((((2 * ARRAY.pi) * 12) * (xx + yy)) / size))
patterns ← [("Vertical stripes", vertical), ("Horizontal stripes", horizontal), ("Diagonal stripes",
    diagonal)]
(fig, axes) ← PLOT.subplots(3, 2, figsize = (10, 13))
FOR (row, (name, pattern)) IN ENUMERATE(patterns) DO
    F ← compute_centered_fft2(pattern)
    CALL display_grayscale_image(axes[(row, 0)], pattern, name)
    CALL display_grayscale_image(axes[(row, 1)], compute_log_spectrum_magnitude(F), FORMAT("{0} — spectrum" ,
        name))
END FOR
CALL fig.tight_layout()
CALL save_output_figure(fig, "04_orientation_spectra.png")
CALL PLOT.show()
frequency_domain_input_files ← LIST[path FOR path IN DATASETS["Fourier"] WHERE (path ≠ HOUSE_PATH)]
REQUIRE frequency_domain_input_files OTHERWISE "At least one additional Fourier image is required."
(fig, axes) ← PLOT.subplots(LENGTH(frequency_domain_input_files), 2, figsize = (12,
    (4 * LENGTH(frequency_domain_input_files))), squeeze = FALSE)
FOR (row, image_path) IN ENUMERATE(frequency_domain_input_files) DO
    image ← load_grayscale_image(image_path)
    F ← compute_centered_fft2(image)
    CALL display_grayscale_image(axes[(row, 0)], image, image_path.name)
    CALL display_grayscale_image(axes[(row, 1)], compute_log_spectrum_magnitude(F), FORMAT("{0} — spectrum" ,
        image_path.name))
END FOR
CALL fig.tight_layout()
CALL save_output_figure(fig, "05_dataset_spectra.png")
CALL PLOT.show()

# Algorithm 8 — 5. Inverse FFT and Reconstruction
reconstructed_house ← reconstruct_from_centered_fft2(house_centered_spectrum)
reconstruction_absolute_error ← ARRAY.abs((CAST(house, ARRAY.float64) − reconstructed_house))
allowed_reconstruction_error ← compute_fft_roundtrip_tolerance(house)
CALL EMIT("Maximum reconstruction error:", FORMAT("{0:.6e}" , MAX(reconstruction_absolute_error)))
CALL EMIT("Float-aware tolerance:", FORMAT("{0:.6e}" , allowed_reconstruction_error))
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], house, "Original")
CALL display_grayscale_image(axes[1], reconstructed_house, "IFFT reconstruction")
CALL display_grayscale_image(axes[2], reconstruction_absolute_error, "Absolute error")
CALL fig.tight_layout()
CALL save_output_figure(fig, "06_reconstruction.png")
CALL PLOT.show()

# Algorithm 9 — 6. Magnitude–Phase Analysis
CAT_PATH ← select_input_image("PhaseMag", "cat")
WOLF_PATH ← select_input_image("PhaseMag", "wolf")
cat ← load_grayscale_image(CAT_PATH)
wolf ← load_grayscale_image(WOLF_PATH)
common_size ← (256, 256)
resized_cat_image ← ARRAY.asarray(IMAGE.fromarray(CAST(cat, ARRAY.uint8)).resize(common_size),
    dtype = ARRAY.float32)
resized_wolf_image ← ARRAY.asarray(IMAGE.fromarray(CAST(wolf, ARRAY.uint8)).resize(common_size),
    dtype = ARRAY.float32)
cat_centered_spectrum ← ARRAY.fft.fft2(resized_cat_image)
wolf_centered_spectrum ← ARRAY.fft.fft2(resized_wolf_image)
cat_magnitude_spectrum ← ARRAY.abs(cat_centered_spectrum)
cat_phase_spectrum ← ARRAY.angle(cat_centered_spectrum)
wolf_magnitude_spectrum ← ARRAY.abs(wolf_centered_spectrum)
wolf_phase_spectrum ← ARRAY.angle(wolf_centered_spectrum)
cat_magnitude_wolf_phase_image ← ARRAY.real(ARRAY.fft.ifft2((cat_magnitude_spectrum * ARRAY.exp((1i * wolf_phase_spectrum)))))
wolf_magnitude_cat_phase_image ← ARRAY.real(ARRAY.fft.ifft2((wolf_magnitude_spectrum * ARRAY.exp((1i * cat_phase_spectrum)))))
(fig, axes) ← PLOT.subplots(2, 2, figsize = (10, 10))
CALL display_grayscale_image(axes[(0, 0)], resized_cat_image, CAT_PATH.stem)
CALL display_grayscale_image(axes[(0, 1)], resized_wolf_image, WOLF_PATH.stem)
CALL display_grayscale_image(axes[(1, 0)], cat_magnitude_wolf_phase_image,
    FORMAT("{0} magnitude + {1} phase" , CAT_PATH.stem, WOLF_PATH.stem))
CALL display_grayscale_image(axes[(1, 1)], wolf_magnitude_cat_phase_image,
    FORMAT("{0} magnitude + {1} phase" , WOLF_PATH.stem, CAT_PATH.stem))
CALL fig.tight_layout()
CALL save_output_figure(fig, "07_phase_magnitude_swap.png")
CALL PLOT.show()
magnitude_only ← ARRAY.real(ARRAY.fft.ifft2((cat_magnitude_spectrum * ARRAY.exp((1i * ARRAY.zeros_like(cat_phase_spectrum))))))
phase_only ← ARRAY.real(ARRAY.fft.ifft2((ARRAY.ones_like(cat_magnitude_spectrum) * ARRAY.exp((1i * cat_phase_spectrum)))))
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], resized_cat_image, "Original")
CALL display_grayscale_image(axes[1], magnitude_only, "Magnitude only")
CALL display_grayscale_image(axes[2], phase_only, "Phase only")
CALL fig.tight_layout()
CALL save_output_figure(fig, "08_phase_only_magnitude_only.png")
CALL PLOT.show()

# Algorithm 10 — 7. Frequency-Domain Filtering
PROCEDURE apply_frequency_domain_filter(image, H)
    image ← ARRAY.asarray(image, dtype = ARRAY.float64)
    H ← ARRAY.asarray(H, dtype = ARRAY.float64)
    IF (image.shape ≠ H.shape) THEN
        RAISE ValueError(FORMAT("Image shape {0} and filter shape {1} must match." , image.shape, H.shape))
    END IF
    centered_spectrum ← compute_centered_fft2(image)
    filtered_spectrum ← (centered_spectrum * H)
    result ← reconstruct_from_centered_fft2(filtered_spectrum)
    RETURN (result, centered_spectrum, filtered_spectrum)
END PROCEDURE
identity_filter ← ARRAY.ones(house.shape, dtype = ARRAY.float64)
(identity_result, identity_spectrum, identity_filtered_spectrum) ← apply_frequency_domain_filter(house,
    identity_filter)
identity_error ← MAX(ARRAY, ARRAY.abs((identity_result − CAST(house, ARRAY.float64))))
identity_tolerance ← compute_fft_roundtrip_tolerance(house)
REQUIRE (identity_result.shape = house.shape) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isfinite(identity_result).all() OTHERWISE STOP WITH ValidationError
REQUIRE (identity_error ≤ identity_tolerance) OTHERWISE STOP WITH ValidationError
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], house, "Input image")
CALL display_grayscale_image(axes[1], compute_log_spectrum_magnitude(identity_spectrum),
    "Centered spectrum")
CALL display_grayscale_image(axes[2], identity_result,
    FORMAT("Identity-filter reconstruction\nmax error={0:.2e}" , identity_error))
CALL fig.tight_layout()
CALL save_output_figure(fig, "07_frequency_filter_pipeline.png")
CALL PLOT.show()
CALL EMIT(FORMAT("Frequency-domain pipeline: PASS | maximum identity error = {0:.3e} | tolerance = {1:.3e}" ,
    identity_error, identity_tolerance))

# Algorithm 11 — 8. Frequency Distance Grid
PROCEDURE build_frequency_distance_grid(shape)
    (rows, cols) ← shape
    cy ← (rows DIV 2)
    cx ← (cols DIV 2)
    (y, x) ← ARRAY.ogrid[(:rows, :cols)]
    RETURN ARRAY.sqrt((((y − cy) ^ 2) + ((x − cx) ^ 2)))
END PROCEDURE
D_house ← build_frequency_distance_grid(house.shape)
CALL PLOT.figure(figsize = (6, 5))
CALL PLOT.imshow(D_house, cmap = "viridis")
CALL PLOT.title("Distance from frequency origin")
CALL PLOT.colorbar(label = "Frequency-bin distance")
CALL PLOT.axis("off")
CALL PLOT.show()

# Algorithm 12 — 9. Ideal, Gaussian, and Butterworth Low-Pass Filters
PROCEDURE build_ideal_low_pass_filter(shape, cutoff)
    D ← build_frequency_distance_grid(shape)
    RETURN CAST((D ≤ cutoff), ARRAY.float32)
END PROCEDURE
PROCEDURE build_gaussian_low_pass_filter(shape, cutoff)
    D ← build_frequency_distance_grid(shape)
    RETURN ARRAY.exp(((−(D ^ 2)) / (2 * (cutoff ^ 2))))
END PROCEDURE
PROCEDURE build_butterworth_low_pass_filter(shape, cutoff, order = 2)
    D ← build_frequency_distance_grid(shape)
    RETURN (1.0 / (1.0 + ((D / MAX(REAL(cutoff), 1e-12)) ^ (2 * order))))
END PROCEDURE
cutoff ← 30
H_ideal_lp ← build_ideal_low_pass_filter(house.shape, cutoff)
H_gaussian_lp ← build_gaussian_low_pass_filter(house.shape, cutoff)
H_butterworth_lp ← build_butterworth_low_pass_filter(house.shape, cutoff, order = 2)
(house_ideal_lp, _, _) ← apply_frequency_domain_filter(house, H_ideal_lp)
(house_gaussian_lp, _, _) ← apply_frequency_domain_filter(house, H_gaussian_lp)
(house_butterworth_lp, _, _) ← apply_frequency_domain_filter(house, H_butterworth_lp)
(fig, axes) ← PLOT.subplots(2, 3, figsize = (15, 10))
CALL display_grayscale_image(axes[(0, 0)], H_ideal_lp, "Ideal LPF")
CALL display_grayscale_image(axes[(0, 1)], H_gaussian_lp, "Gaussian LPF")
CALL display_grayscale_image(axes[(0, 2)], H_butterworth_lp, "Butterworth LPF")
CALL display_grayscale_image(axes[(1, 0)], house_ideal_lp, "Ideal result")
CALL display_grayscale_image(axes[(1, 1)], house_gaussian_lp, "Gaussian result")
CALL display_grayscale_image(axes[(1, 2)], house_butterworth_lp, "Butterworth result")
CALL fig.tight_layout()
CALL save_output_figure(fig, "09_lpf_comparison.png")
CALL PLOT.show()
lpf_quality_metrics ← {"Ideal" ↦ {"mse" ↦ mean_squared_error(house, house_ideal_lp),
    "psnr" ↦ peak_signal_to_noise_ratio(house, house_ideal_lp),
    "edge_strength" ↦ compute_mean_gradient_magnitude(house_ideal_lp)},
    "Gaussian" ↦ {"mse" ↦ mean_squared_error(house, house_gaussian_lp),
    "psnr" ↦ peak_signal_to_noise_ratio(house, house_gaussian_lp),
    "edge_strength" ↦ compute_mean_gradient_magnitude(house_gaussian_lp)},
    "Butterworth n=2" ↦ {"mse" ↦ mean_squared_error(house, house_butterworth_lp),
    "psnr" ↦ peak_signal_to_noise_ratio(house, house_butterworth_lp),
    "edge_strength" ↦ compute_mean_gradient_magnitude(house_butterworth_lp)}}
orders ← [1, 2, 4, 8]
butterworth_order_results ← []
(fig, axes) ← PLOT.subplots(LENGTH(orders), 2, figsize = (11, 15))
FOR (row, order) IN ENUMERATE(orders) DO
    H ← build_butterworth_low_pass_filter(house.shape, cutoff, order = order)
    (result, _, _) ← apply_frequency_domain_filter(house, H)
    CALL APPEND(butterworth_order_results, {"order" ↦ order, "mse" ↦ mean_squared_error(house, result),
        "edge_strength" ↦ compute_mean_gradient_magnitude(result)})
    CALL display_grayscale_image(axes[(row, 0)], H, FORMAT("Butterworth n={0}" , order))
    CALL display_grayscale_image(axes[(row, 1)], result, FORMAT("Result n={0}" , order))
END FOR
CALL fig.tight_layout()
CALL save_output_figure(fig, "10_butterworth_orders.png")
CALL PLOT.show()

# Algorithm 13 — 10. Ringing and the Gibbs Phenomenon
square ← ARRAY.zeros((256, 256), dtype = ARRAY.float32)
square[(64:192, 64:192)] ← 255.0
ring_cutoff ← 22
Hi ← build_ideal_low_pass_filter(square.shape, ring_cutoff)
Hg ← build_gaussian_low_pass_filter(square.shape, ring_cutoff)
Hb ← build_butterworth_low_pass_filter(square.shape, ring_cutoff, order = 2)
(square_i, _, _) ← apply_frequency_domain_filter(square, Hi)
(square_g, _, _) ← apply_frequency_domain_filter(square, Hg)
(square_b, _, _) ← apply_frequency_domain_filter(square, Hb)
center_row ← (square.shape[0] DIV 2)
(fig, axes) ← PLOT.subplots(2, 2, figsize = (13, 10))
CALL display_grayscale_image(axes[(0, 0)], square, "Original square")
CALL display_grayscale_image(axes[(0, 1)], square_i, "Ideal LPF — ringing")
CALL axes[(1, 0)].plot(square[center_row], label = "Original")
CALL axes[(1, 0)].plot(square_i[center_row], label = "Ideal")
CALL axes[(1, 0)].plot(square_g[center_row], label = "Gaussian")
CALL axes[(1, 0)].plot(square_b[center_row], label = "Butterworth")
CALL axes[(1, 0)].set_title("Intensity profile through edge")
CALL axes[(1, 0)].legend()
CALL display_grayscale_image(axes[(1, 1)], ARRAY.abs((square_i − square)), "Ideal LPF difference")
CALL fig.tight_layout()
CALL save_output_figure(fig, "11_ringing.png")
CALL PLOT.show()
ringing_by_method ← {"Ideal" ↦ measure_ringing_overshoot(square_i),
    "Gaussian" ↦ measure_ringing_overshoot(square_g), "Butterworth n=2" ↦ measure_ringing_overshoot(square_b)}
butterworth_order_ringing ← []
FOR order IN orders DO
    H_order ← build_butterworth_low_pass_filter(square.shape, ring_cutoff, order = order)
    (square_order, _, _) ← apply_frequency_domain_filter(square, H_order)
    CALL APPEND(butterworth_order_ringing, {"order" ↦ order,
        "ringing" ↦ measure_ringing_overshoot(square_order)})
END FOR

# Algorithm 14 — 11. High-Pass Filtering
PROCEDURE build_high_pass_from_low_pass(H_low)
    RETURN (1.0 − H_low)
END PROCEDURE
H_ideal_hp ← build_high_pass_from_low_pass(H_ideal_lp)
H_gaussian_hp ← build_high_pass_from_low_pass(H_gaussian_lp)
H_butterworth_hp ← build_high_pass_from_low_pass(H_butterworth_lp)
(house_i_hp, _, _) ← apply_frequency_domain_filter(house, H_ideal_hp)
(house_g_hp, _, _) ← apply_frequency_domain_filter(house, H_gaussian_hp)
(house_b_hp, _, _) ← apply_frequency_domain_filter(house, H_butterworth_hp)
(fig, axes) ← PLOT.subplots(2, 3, figsize = (15, 10))
CALL display_grayscale_image(axes[(0, 0)], H_ideal_hp, "Ideal HPF")
CALL display_grayscale_image(axes[(0, 1)], H_gaussian_hp, "Gaussian HPF")
CALL display_grayscale_image(axes[(0, 2)], H_butterworth_hp, "Butterworth HPF")
CALL display_grayscale_image(axes[(1, 0)], normalize_to_unit_interval(house_i_hp), "Ideal HP response")
CALL display_grayscale_image(axes[(1, 1)], normalize_to_unit_interval(house_g_hp), "Gaussian HP response")
CALL display_grayscale_image(axes[(1, 2)], normalize_to_unit_interval(house_b_hp),
    "Butterworth HP response")
CALL fig.tight_layout()
CALL save_output_figure(fig, "12_high_pass.png")
CALL PLOT.show()

# Algorithm 15 — 12. High-Boost Sharpening
high_detail ← house_g_hp
high_boost_gains ← [0.5, 1.0, 1.5, 2.0]
high_boost_results ← []
(fig, axes) ← PLOT.subplots(1, (LENGTH(high_boost_gains) + 1),
    figsize = ((4 * (LENGTH(high_boost_gains) + 1)), 5))
CALL display_grayscale_image(axes[0], house, "Original")
FOR (col, gain) IN ENUMERATE(high_boost_gains, start = 1) DO
    boosted_unclipped ← (CAST(house, ARRAY.float64) + (gain * high_detail))
    clipped_mask ← ((boosted_unclipped < 0.0) OR_EACH (boosted_unclipped > 255.0))
    boosted ← ARRAY.clip(boosted_unclipped, 0.0, 255.0)
    clipping_percent ← (100.0 * MEAN(ARRAY, clipped_mask))
    CALL APPEND(high_boost_results, {"gain" ↦ gain, "clipping_percent" ↦ clipping_percent,
        "edge_strength" ↦ compute_mean_gradient_magnitude(boosted)})
    CALL display_grayscale_image(axes[col], boosted, FORMAT("k={0:.1f}\nclipped={1:.2f}%" , gain,
        clipping_percent))
END FOR
CALL fig.tight_layout()
CALL save_output_figure(fig, "13_high_boost.png")
CALL PLOT.show()
k ← 1.2
high_boost ← ARRAY.clip((CAST(house, ARRAY.float64) + (k * high_detail)), 0.0, 255.0)

# Algorithm 16 — 13. Convolution Theorem
conv_image ← CAST(house[(:96, :96)], ARRAY.float64)
conv_kernel ← ARRAY.ones((5, 5), dtype = ARRAY.float64)
conv_kernel ← (conv_kernel / SUM(conv_kernel))
pad_y ← (conv_kernel.shape[0] DIV 2)
pad_x ← (conv_kernel.shape[1] DIV 2)
spatial_padded ← ARRAY.pad(conv_image, ((pad_y, pad_y), (pad_x, pad_x)), mode = "constant")
windows ← ARRAY.lib.stride_tricks.sliding_window_view(spatial_padded, conv_kernel.shape)
spatial_conv ← SUM(ARRAY, (windows * conv_kernel[(::(−1), ::(−1))]), axis = ((−2), (−1)))
fft_shape ← (((conv_image.shape[0] + conv_kernel.shape[0]) − 1),
    ((conv_image.shape[1] + conv_kernel.shape[1]) − 1))
fft_full ← ARRAY.real(ARRAY.fft.ifft2((ARRAY.fft.fft2(conv_image,
    s = fft_shape) * ARRAY.fft.fft2(conv_kernel, s = fft_shape))))
fft_conv ← fft_full[(pad_y:(pad_y + conv_image.shape[0]), pad_x:(pad_x + conv_image.shape[1]))]
conv_error ← ARRAY.abs((spatial_conv − fft_conv))
REQUIRE (spatial_conv.shape = fft_conv.shape) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isfinite(fft_conv).all() OTHERWISE STOP WITH ValidationError
REQUIRE (MAX(conv_error) < 1e-08) OTHERWISE STOP WITH ValidationError
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], spatial_conv, "Direct spatial convolution")
CALL display_grayscale_image(axes[1], fft_conv, "Zero-padded FFT convolution")
CALL display_grayscale_image(axes[2], conv_error, FORMAT("Absolute difference\nmax={0:.2e}" ,
    MAX(conv_error)))
CALL fig.tight_layout()
CALL save_output_figure(fig, "13_convolution_validation.png")
CALL PLOT.show()
CALL EMIT(FORMAT("Convolution validation: PASS | maximum absolute difference = {0:.3e}" , MAX(conv_error)))

# Algorithm 17 — 14. Band-Pass and Band-Reject Filters
PROCEDURE build_ideal_band_pass_filter(shape, low_cutoff, high_cutoff)
    D ← build_frequency_distance_grid(shape)
    RETURN CAST(((D ≥ low_cutoff) AND_EACH (D ≤ high_cutoff)), ARRAY.float32)
END PROCEDURE
H_band ← build_ideal_band_pass_filter(house.shape, 15, 55)
H_band_reject ← (1.0 − H_band)
(house_band, _, _) ← apply_frequency_domain_filter(house, H_band)
(house_band_reject, _, _) ← apply_frequency_domain_filter(house, H_band_reject)
(fig, axes) ← PLOT.subplots(2, 2, figsize = (11, 10))
CALL display_grayscale_image(axes[(0, 0)], H_band, "Band-pass mask")
CALL display_grayscale_image(axes[(0, 1)], normalize_to_unit_interval(house_band), "Band-pass content")
CALL display_grayscale_image(axes[(1, 0)], H_band_reject, "Band-reject mask")
CALL display_grayscale_image(axes[(1, 1)], house_band_reject, "Band-reject result")
CALL fig.tight_layout()
CALL save_output_figure(fig, "14_band_filters.png")
CALL PLOT.show()

# Algorithm 18 — 15. Periodic Interference Analysis
interference_files ← LIST[path FOR path IN DATASETS["Frequency"] WHERE ANY(STREAM[(token IN path.stem.lower()) FOR token IN ("interference",
    "moire")])]
REQUIRE interference_files OTHERWISE "No periodic-interference images were discovered."
(fig, axes) ← PLOT.subplots(LENGTH(interference_files), 2, figsize = (12, (5 * LENGTH(interference_files))),
    squeeze = FALSE)
FOR (row, image_path) IN ENUMERATE(interference_files) DO
    image ← load_grayscale_image(image_path)
    F ← compute_centered_fft2(image)
    CALL display_grayscale_image(axes[(row, 0)], image, image_path.name)
    CALL display_grayscale_image(axes[(row, 1)], compute_log_spectrum_magnitude(F), FORMAT("{0} — spectrum" ,
        image_path.name))
END FOR
CALL fig.tight_layout()
CALL save_output_figure(fig, "15_periodic_noise_spectra.png")
CALL PLOT.show()

# Algorithm 19 — 16. Spectral Peak Detection
PROCEDURE find_strongest_spectral_peaks(centered_spectrum, number_of_peaks = 8, center_exclusion_radius = 20,
    min_separation = 10)
    magnitude ← COPY(compute_log_spectrum_magnitude(centered_spectrum))
    (rows, cols) ← magnitude.shape
    (cy, cx) ← ((rows DIV 2), (cols DIV 2))
    (yy, xx) ← ARRAY.ogrid[(:rows, :cols)]
    center_mask ← ((((yy − cy) ^ 2) + ((xx − cx) ^ 2)) ≤ (center_exclusion_radius ^ 2))
    magnitude[center_mask] ← (−ARRAY.inf)
    flat_order ← ARRAY.argsort(FLATTEN(magnitude))[::(−1)]
    selected ← []
    FOR flat_index IN flat_order DO
        (y, x) ← ARRAY.unravel_index(flat_index, magnitude.shape)
        separated ← ALL(STREAM[((((y − py) ^ 2) + ((x − px) ^ 2)) ≥ (min_separation ^ 2)) FOR (py,
            px) IN selected])
        IF separated THEN
            CALL APPEND(selected, (y, x))
        END IF
        IF (LENGTH(selected) ≥ number_of_peaks) THEN
            BREAK
        END IF
    END FOR
    RETURN selected
END PROCEDURE

# Algorithm 20 — 17. Notch-Reject Filtering
PROCEDURE build_notch_reject_mask(shape, peak_locations, radius = 5)
    (rows, cols) ← shape
    (cy, cx) ← ((rows DIV 2), (cols DIV 2))
    (yy, xx) ← ARRAY.ogrid[(:rows, :cols)]
    mask ← ARRAY.ones(shape, dtype = ARRAY.float32)
    FOR (py, px) IN peak_locations DO
        d1 ← (((yy − py) ^ 2) + ((xx − px) ^ 2))
        mask[(d1 ≤ (radius ^ 2))] ← 0.0
        sym_y ← ((2 * cy) − py)
        sym_x ← ((2 * cx) − px)
        d2 ← (((yy − sym_y) ^ 2) + ((xx − sym_x) ^ 2))
        mask[(d2 ≤ (radius ^ 2))] ← 0.0
    END FOR
    RETURN mask
END PROCEDURE
ASTRONAUT_PATH ← select_input_image("Frequency", "astronaut")
astronaut ← load_grayscale_image(ASTRONAUT_PATH)
F_astronaut ← compute_centered_fft2(astronaut)
astronaut_peaks ← find_strongest_spectral_peaks(F_astronaut, number_of_peaks = 8,
    center_exclusion_radius = 25, min_separation = 12)
notch_radii ← [2, 4, 6, 8]
notch_radius_results ← []
astronaut_energy ← SUM(ARRAY, (ARRAY.abs(F_astronaut) ^ 2))
FOR radius IN notch_radii DO
    H_radius ← build_notch_reject_mask(astronaut.shape, astronaut_peaks[:4], radius = radius)
    removed_energy ← SUM(ARRAY, ((ARRAY.abs(F_astronaut) ^ 2) * (1.0 − H_radius)))
    CALL APPEND(notch_radius_results, {"radius" ↦ radius,
        "removed_energy_percent" ↦ ((100.0 * removed_energy) / MAX(astronaut_energy, 1e-12))})
END FOR
selected_notch_radius ← 4
astronaut_notch ← build_notch_reject_mask(astronaut.shape, astronaut_peaks[:4],
    radius = selected_notch_radius)
astronaut_filtered ← reconstruct_from_centered_fft2((F_astronaut * astronaut_notch))
(fig, axes) ← PLOT.subplots(2, 2, figsize = (12, 10))
CALL display_grayscale_image(axes[(0, 0)], astronaut, ASTRONAUT_PATH.name)
CALL display_grayscale_image(axes[(0, 1)], compute_log_spectrum_magnitude(F_astronaut), "Original spectrum")
CALL display_grayscale_image(axes[(1, 0)], astronaut_notch, FORMAT("Notch mask — r={0}" ,
    selected_notch_radius))
CALL display_grayscale_image(axes[(1, 1)], astronaut_filtered, "After notch filtering")
CALL fig.tight_layout()
CALL save_output_figure(fig, "16_notch_filter.png")
CALL PLOT.show()
CALL EMIT("Candidate peaks:", astronaut_peaks)

# Algorithm 21 — 18. Moiré Removal
MOIRE_PATH ← select_input_image("Frequency", "moire")
car_moire ← load_grayscale_image(MOIRE_PATH)
F_car ← compute_centered_fft2(car_moire)
car_peaks ← find_strongest_spectral_peaks(F_car, number_of_peaks = 10, center_exclusion_radius = 30,
    min_separation = 12)
car_notch ← build_notch_reject_mask(car_moire.shape, car_peaks[:6], radius = 4)
car_filtered ← reconstruct_from_centered_fft2((F_car * car_notch))
(fig, axes) ← PLOT.subplots(1, 3, figsize = (16, 5))
CALL display_grayscale_image(axes[0], car_moire, MOIRE_PATH.name)
CALL display_grayscale_image(axes[1], compute_log_spectrum_magnitude(F_car), "Moiré spectrum")
CALL display_grayscale_image(axes[2], car_filtered, "Notch-filtered result")
CALL fig.tight_layout()
CALL save_output_figure(fig, "17_moire_removal.png")
CALL PLOT.show()

# Algorithm 22 — 19. Low-Frequency Illumination Correction
SPOTSHADE_PATH ← select_input_image("Frequency", "spotshade")
spotshade ← load_grayscale_image(SPOTSHADE_PATH)
illumination_filter ← build_gaussian_low_pass_filter(spotshade.shape, cutoff = 18)
(illumination, _, _) ← apply_frequency_domain_filter(spotshade, illumination_filter)
epsilon ← 1e-06
corrected ← (spotshade / (illumination + epsilon))
corrected ← normalize_to_unit_interval(corrected)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (15, 5))
CALL display_grayscale_image(axes[0], spotshade, SPOTSHADE_PATH.name)
CALL display_grayscale_image(axes[1], illumination, "Estimated illumination")
CALL display_grayscale_image(axes[2], corrected, "Normalized image")
CALL fig.tight_layout()
CALL save_output_figure(fig, "18_shading_correction.png")
CALL PLOT.show()

# Algorithm 23 — 20. Cutoff Sensitivity
cutoffs ← [10, 25, 60, 120]
cutoff_sensitivity_results ← []
(fig, axes) ← PLOT.subplots(2, LENGTH(cutoffs), figsize = ((4 * LENGTH(cutoffs)), 8))
FOR (col, current_cutoff) IN ENUMERATE(cutoffs) DO
    H ← build_gaussian_low_pass_filter(house.shape, current_cutoff)
    (filtered, _, _) ← apply_frequency_domain_filter(house, H)
    CALL APPEND(cutoff_sensitivity_results, {"cutoff" ↦ current_cutoff, "mse" ↦ mean_squared_error(house,
        filtered), "edge_strength" ↦ compute_mean_gradient_magnitude(filtered)})
    CALL display_grayscale_image(axes[(0, col)], H, FORMAT("Gaussian LPF\nD0={0}" , current_cutoff))
    CALL display_grayscale_image(axes[(1, col)], filtered, FORMAT("Result\nD0={0}" , current_cutoff))
END FOR
CALL fig.tight_layout()
CALL save_output_figure(fig, "19_cutoff_sensitivity.png")
CALL PLOT.show()

# Algorithm 24 — 21. Quantitative Checks
CALL EMIT("Low-pass comparison at cutoff D0=30")
FOR (name, metrics) IN ITEMS(lpf_quality_metrics) DO
    CALL EMIT(FORMAT("{0:20s} MSE={1:10.3f} PSNR={2:7.2f} dB Edge={3:8.3f}" , name, metrics["mse"],
        metrics["psnr"], metrics["edge_strength"]))
END FOR

# Algorithm 25 — 22. Validation Checks
validation_reconstruction ← reconstruct_from_centered_fft2(compute_centered_fft2(house))
validation_error ← ARRAY.abs((CAST(house, ARRAY.float64) − validation_reconstruction))
validation_tolerance ← compute_fft_roundtrip_tolerance(house)
REQUIRE (validation_reconstruction.shape = house.shape) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isfinite(validation_reconstruction).all() OTHERWISE STOP WITH ValidationError
REQUIRE (MAX(validation_error) ≤ validation_tolerance) OTHERWISE FORMAT("FFT/IFFT round-trip error exceeded the dtype-aware tolerance: {0:.6e} > {1:.6e}" ,
    MAX(validation_error), validation_tolerance)
filters_to_check ← [H_ideal_lp, H_gaussian_lp, H_butterworth_lp, H_ideal_hp, H_gaussian_hp,
    H_butterworth_hp]
FOR H IN filters_to_check DO
    REQUIRE (H.shape = house.shape) OTHERWISE STOP WITH ValidationError
    REQUIRE ARRAY.isfinite(H).all() OTHERWISE STOP WITH ValidationError
    REQUIRE (MIN(H) ≥ 0) OTHERWISE STOP WITH ValidationError
    REQUIRE (MAX(H) ≤ 1) OTHERWISE STOP WITH ValidationError
END FOR
center ← ((house.shape[0] DIV 2), (house.shape[1] DIV 2))
REQUIRE ARRAY.isclose(H_ideal_lp[center], 1.0) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(H_gaussian_lp[center], 1.0) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(H_butterworth_lp[center], 1.0) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(H_ideal_hp[center], 0.0) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(H_gaussian_hp[center], 0.0) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(H_butterworth_hp[center], 0.0) OTHERWISE STOP WITH ValidationError
CALL EMIT("PASS — reconstruction and filter sanity checks")
CALL EMIT("Round-trip max error:", FORMAT("{0:.6e}" , MAX(validation_error)))
CALL EMIT("Accepted tolerance:", FORMAT("{0:.6e}" , validation_tolerance))

# Algorithm 26 — 23. Failure Modes and Diagnostic Signatures
raw_dynamic_range ← (MAX(ARRAY, ARRAY.abs(house_centered_spectrum)) / MAX(ARRAY.median(ARRAY.abs(house_centered_spectrum)),
    1e-12))
failure_diagnostics ← {"FFT raw dynamic-range ratio" ↦ raw_dynamic_range,
    "FFT/IFFT round-trip max error" ↦ MAX(validation_error),
    "FFT/IFFT accepted tolerance" ↦ validation_tolerance, "Convolution max error" ↦ MAX(conv_error),
    "Ideal LPF ringing overshoot" ↦ ringing_by_method["Ideal"],
    "Gaussian LPF ringing overshoot" ↦ ringing_by_method["Gaussian"],
    "High-boost max clipping (%)" ↦ MAX(STREAM[row["clipping_percent"] FOR row IN high_boost_results])}
CALL EMIT("Failure-mode diagnostic summary")
FOR (name, value) IN ITEMS(failure_diagnostics) DO
    CALL EMIT(FORMAT("  {0:36s}: {1:.6e}" , name, value))
END FOR
CALL EMIT()
CALL EMIT("Interpretation: large raw spectral dynamic range justifies log-magnitude visualization; round-trip and convolution errors verify numerical correctness; ringing and clipping quantify the main artifact risks already observed above.")

# Algorithm 27 — 24. Parameter Sensitivity and Controlled Experiments
CALL EMIT("Controlled parameter sensitivity summary")
CALL EMIT("\nGaussian cutoff sweep")
FOR row IN cutoff_sensitivity_results DO
    CALL EMIT(FORMAT("  D0={0:3d} | MSE={1:10.3f} | Edge={2:8.3f}" , row["cutoff"], row["mse"],
        row["edge_strength"]))
END FOR
CALL EMIT("\nButterworth order sweep")
ringing_lookup ← MAP{row["order"] ↦ row["ringing"] FOR row IN butterworth_order_ringing}
FOR row IN butterworth_order_results DO
    CALL EMIT(FORMAT("  n={0:2d} | MSE={1:10.3f} | Edge={2:8.3f} | Ringing={3:8.3f}" , row["order"],
        row["mse"], row["edge_strength"], ringing_lookup[row["order"]]))
END FOR
CALL EMIT("\nHigh-boost gain sweep")
FOR row IN high_boost_results DO
    CALL EMIT(FORMAT("  k={0:.1f} | Clipped={1:7.3f}% | Edge={2:8.3f}" , row["gain"],
        row["clipping_percent"], row["edge_strength"]))
END FOR
CALL EMIT("\nNotch-radius sweep")
FOR row IN notch_radius_results DO
    CALL EMIT(FORMAT("  r={0:2d} | Removed spectral energy={1:.6f}%" , row["radius"],
        row["removed_energy_percent"]))
END FOR

# Algorithm 28 — 25. Method Selection and Technical Discussion
method_selection_results ← []
FOR (method, metrics) IN ITEMS(lpf_quality_metrics) DO
    CALL APPEND(method_selection_results, {"method" ↦ method, "mse" ↦ metrics["mse"],
        "psnr" ↦ metrics["psnr"], "edge_strength" ↦ metrics["edge_strength"],
        "ringing" ↦ ringing_by_method[method]})
END FOR
CALL EMIT("Method                 MSE        PSNR(dB)   EdgeStrength   Ringing")
FOR row IN method_selection_results DO
    CALL EMIT(FORMAT("{0:20s} {1:10.3f} {2:10.2f} {3:14.3f} {4:9.3f}" , row["method"], row["mse"],
        row["psnr"], row["edge_strength"], row["ringing"]))
END FOR
lowest_change ← MIN(method_selection_results, key = FUNCTION(row) ↦ row["mse"])
lowest_ringing ← MIN(method_selection_results, key = FUNCTION(row) ↦ row["ringing"])
strongest_edges ← MAX(method_selection_results, key = FUNCTION(row) ↦ row["edge_strength"])
CALL EMIT()
CALL EMIT("Lowest numerical change :", lowest_change["method"])
CALL EMIT("Lowest measured ringing :", lowest_ringing["method"])
CALL EMIT("Highest retained edge strength:", strongest_edges["method"])

# Algorithm 29 — 26. Integrated Frequency-Domain Workflow
PROCEDURE run_frequency_filtering_workflow(image_path, filter_family = "butterworth", cutoff = 30,
    order = 2)
    image ← load_grayscale_image(image_path)
    builders ← {"ideal" ↦ FUNCTION() ↦ build_ideal_low_pass_filter(image.shape, cutoff),
        "gaussian" ↦ FUNCTION() ↦ build_gaussian_low_pass_filter(image.shape, cutoff),
        "butterworth" ↦ FUNCTION() ↦ build_butterworth_low_pass_filter(image.shape, cutoff, order = order)}
    IF (filter_family NOT IN builders) THEN
        RAISE ValueError(("filter_family must be one of: " + ", ".join(builders)))
    END IF
    H ← builders[filter_family]()
    (result, F_input, F_filtered) ← apply_frequency_domain_filter(image, H)
    REQUIRE (result.shape = image.shape) OTHERWISE STOP WITH ValidationError
    REQUIRE (H.shape = image.shape) OTHERWISE STOP WITH ValidationError
    REQUIRE ARRAY.isfinite(result).all() OTHERWISE STOP WITH ValidationError
    REQUIRE ARRAY.isfinite(H).all() OTHERWISE STOP WITH ValidationError
    REQUIRE (MIN(H) ≥ 0.0) OTHERWISE STOP WITH ValidationError
    REQUIRE (MAX(H) ≤ 1.0) OTHERWISE STOP WITH ValidationError
    report ← {"image" ↦ image_path.name, "shape" ↦ image.shape, "filter_family" ↦ filter_family,
        "cutoff" ↦ cutoff, "order" ↦ (IF (filter_family = "butterworth") THEN order ELSE NULL),
        "mse_vs_input" ↦ mean_squared_error(image, result),
        "psnr_vs_input_db" ↦ peak_signal_to_noise_ratio(image, result), "filter_min" ↦ REAL(MIN(H)),
        "filter_max" ↦ REAL(MAX(H))}
    (fig, axes) ← PLOT.subplots(1, 5, figsize = (20, 5))
    CALL display_grayscale_image(axes[0], image, "Input")
    CALL display_grayscale_image(axes[1], compute_log_spectrum_magnitude(F_input), "Centered spectrum")
    CALL display_grayscale_image(axes[2], H, "Selected filter")
    CALL display_grayscale_image(axes[3], compute_log_spectrum_magnitude(F_filtered), "Filtered spectrum")
    CALL display_grayscale_image(axes[4], result, "Reconstructed result")
    CALL fig.tight_layout()
    CALL save_output_figure(fig, "23_integrated_workflow.png")
    CALL PLOT.show()
    RETURN report
END PROCEDURE
final_report ← run_frequency_filtering_workflow(HOUSE_PATH, filter_family = "butterworth", cutoff = 30,
    order = 2)
CALL EMIT("Final workflow configuration")
FOR (key, value) IN ITEMS(final_report) DO
    CALL EMIT(FORMAT("  {0:20s}: {1}" , key, value))
END FOR
REQUIRED_OUTPUTS ← ["01_spatial_frequency.png", "02_fft_1d.png", "03_house_spectrum.png", "04_orientation_spectra.png",
    "05_dataset_spectra.png", "06_reconstruction.png", "07_phase_magnitude_swap.png", "08_phase_only_magnitude_only.png",
    "07_frequency_filter_pipeline.png", "09_lpf_comparison.png", "10_butterworth_orders.png", "11_ringing.png",
    "12_high_pass.png", "13_high_boost.png", "13_convolution_validation.png", "14_band_filters.png", "15_periodic_noise_spectra.png",
    "16_notch_filter.png", "17_moire_removal.png", "18_shading_correction.png", "19_cutoff_sensitivity.png",
    "23_integrated_workflow.png"]
missing_outputs ← LIST[output_name FOR output_name IN REQUIRED_OUTPUTS WHERE (NOT (OUTPUT_DIR / output_name).exists())]
IF missing_outputs THEN
    RAISE FileNotFoundError(("Missing outputs: " + ", ".join(missing_outputs)))
END IF
CALL EMIT(FORMAT("Output validation passed: {0} files." , LENGTH(REQUIRED_OUTPUTS)))

END ALGORITHM
```

## Part II — Step-by-step auditable specification

Each step below is a six-section contract corresponding to the numbered block in Part I:

- **Purpose** — why the step exists.
- **Inputs** — artifacts and shapes required before execution.
- **Procedure** — ordered computation.
- **Produces** — outputs and invariants.
- **Acceptance contract** — numerical or structural gates before the next step.
- **Failure mode** — explicit abort, skip, or diagnostic action; never a silent substitution.

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project kernel and use this notebook as the construction plan for `main.ipynb`.
2. Keep every filtering path traceable from spatial image → centered spectrum → mask → filtered spectrum → inverse FFT.
3. When comparing methods, change only the intended filter or parameter.

**Produces**

Selected environment and a top-to-bottom execution plan.

**Acceptance contract**

Review the local assumptions before using cached notebook outputs.

**Failure mode**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create Setup, Paths, and Dataset Discovery

**Purpose**

Resolve the three datasets without absolute paths.

**Inputs**

Lab root and Fourier, PhaseMag, Frequency folders.

**Procedure**

1. Import `Path`, NumPy, Matplotlib, and PIL.
2. Locate the lab root from the current working directory.
3. Create `data/` and `outputs/figures/` paths.
4. Discover supported images under `Fourier`, `PhaseMag`, and `Frequency`.
5. Fail if a required group contains no usable image.

**Produces**

Dataset inventories and output directory.

**Acceptance contract**

Each group contains supported inputs; semantic selection is deterministic.

**Failure mode**

Stop on a missing group or required image.

## 2. Implement Core Utility Functions

**Purpose**

Centralize FFT, reconstruction and metric conventions.

**Inputs**

Grayscale arrays and paired references.

**Procedure**

1. Create grayscale loading, unit-interval normalization, plotting, and figure-saving helpers.
2. Create `compute_centered_fft2` using `fft2` then `fftshift`.
3. Create inverse reconstruction using `ifftshift` then `ifft2`, retaining the real component.
4. Create log-spectrum magnitude, MSE, PSNR, mean-gradient magnitude, ringing-overshoot, and FFT round-trip tolerance helpers.

**Produces**

Utility functions reused by experiments.

**Acceptance contract**

fftshift pairs with ifftshift; computation retains float values; constant normalization is handled.

**Failure mode**

Correct conventions before building filters.

## 3. Build Spatial-Frequency Demonstrations

**Purpose**

Relate spatial pattern spacing to frequency content.

**Inputs**

Synthetic patterns with declared frequencies/orientations.

**Procedure**

1. Generate controlled stripe or sinusoidal image patterns with known orientation/frequency.
2. Compute centered 2-D spectra for those patterns.
3. Plot spatial pattern beside log-magnitude spectrum.
4. Use this stage to verify how direction and repetition in space appear in frequency.

**Produces**

Pattern and spectrum demonstrations.

**Acceptance contract**

Axes/units and sample spacing are stated.

**Failure mode**

Check sampling effects before interpreting apparent frequencies.

## 4. Validate the 1-D DFT

**Purpose**

Verify the DFT using known signals.

**Inputs**

Sampled sinusoids and sample interval.

**Procedure**

1. Construct a 1-D signal as the sum of two known sinusoids.
2. Compute its FFT and frequency bins.
3. Plot signal and magnitude spectrum.
4. Confirm peaks appear at the injected frequencies before moving to image spectra.

**Produces**

1-D spectra and frequency axis.

**Acceptance contract**

Dominant bins agree with frequencies within bin resolution; conjugate peaks for real signals.

**Failure mode**

Account for leakage if the observation window contains noninteger cycles.

## 5. Compute the 2-D Image Spectrum

**Purpose**

Inspect an image's centred Fourier representation.

**Inputs**

House and additional Fourier images.

**Procedure**

1. Load the selected house image as grayscale float data.
2. Compute its centered 2-D FFT.
3. Compute the log-magnitude spectrum.
4. Save original image and spectrum together.

**Produces**

Complex centred spectra and log-magnitude displays.

**Acceptance contract**

Spectrum shape equals image shape; DC index is (rows//2,cols//2).

**Failure mode**

Keep display log magnitude separate from the complex coefficients.

## 6. Validate Inverse FFT Reconstruction

**Purpose**

Check transform reversibility before filtering.

**Inputs**

Original image and its centred spectrum.

**Procedure**

1. Apply the inverse centered-FFT helper to the untouched spectrum.
2. Compute maximum reconstruction error.
3. Derive tolerance from floating-point precision and image scale.
4. Stop if the FFT/IFFT round trip exceeds tolerance.

**Produces**

Reconstruction and absolute error.

**Acceptance contract**

Compare with the actual dtype-aware tolerance calculated in main.ipynb; shape and finite values preserved.

**Failure mode**

Fix centring/order/dtype errors before proceeding; do not normalize the reconstruction to hide error.

## 7. Analyze Magnitude and Phase

**Purpose**

Separate magnitude and phase contributions.

**Inputs**

Two images prepared at compatible dimensions.

**Procedure**

1. Load the selected PhaseMag images and compute centered spectra.
2. Separate magnitude and unit-complex phase.
3. Reconstruct magnitude-only, phase-only, or magnitude/phase-swapped images required by the lab.
4. Normalize only visualization copies, not the actual Fourier quantities.
5. Save the comparison.

**Produces**

Swapped and isolated magnitude/phase reconstructions.

**Acceptance contract**

Combine magnitude with exp(1j*phase); spectra have matching shapes.

**Failure mode**

Align dimensions before mixing; discuss loss of structure without inventing a fidelity metric.

## 8. Implement the Generic Frequency Filter

**Purpose**

Apply a mask through a consistent transform pair.

**Inputs**

Image and same-shaped transfer function H.

**Procedure**

1. Accept an image and a same-shape frequency mask.
2. Compute centered FFT, multiply elementwise by the mask, and inverse transform.
3. Return reconstructed image plus intermediates needed for diagnostics.
4. Reject shape mismatches and non-finite masks.

**Produces**

Filtered image, input spectrum and filtered spectrum.

**Acceptance contract**

All-ones mask is identity within the notebook's dtype-aware tolerance.

**Failure mode**

Existing helper raises on shape mismatch; investigate nonfinite masks/results.

## 9. Build the Frequency-Distance Grid

**Purpose**

Express cutoff in centred frequency-bin distance.

**Inputs**

Image shape.

**Procedure**

1. Create row/column coordinate grids centered at the spectrum origin.
2. Compute radial distance `D(u,v)` from the center.
3. Reuse this grid for all radial filters.
4. Verify center distance is zero.

**Produces**

Radial distance grid.

**Acceptance contract**

Grid shape matches image; distance at DC is zero.

**Failure mode**

Do not confuse bin radius with physical cycles per unit distance.

## 10. Implement Low-Pass Filter Families

**Purpose**

Compare three low-pass transition profiles.

**Inputs**

Grid, positive cutoff, Butterworth order.

**Procedure**

1. Build the Ideal low-pass mask from a hard cutoff.
2. Build the Gaussian low-pass mask from the Gaussian radial response.
3. Build the Butterworth low-pass mask from cutoff and order.
4. Apply all three to the same image and nominal cutoff.
5. Check low-pass masks are approximately one at the spectrum center.

**Produces**

Ideal, Gaussian and Butterworth masks/results.

**Acceptance contract**

Masks in [0,1], centre near 1, shapes preserved.

**Failure mode**

Require meaningful positive cutoffs; current builders do not uniformly enforce all parameter bounds.

## 11. Study Butterworth Order

**Purpose**

Isolate transition steepness from cutoff.

**Inputs**

Same image/cutoff with varying positive order.

**Procedure**

1. Fix cutoff and vary only Butterworth order.
2. Generate one filtered result per order.
3. Plot results side by side.
4. Use the comparison to isolate transition sharpness from cutoff effects.

**Produces**

Butterworth-order comparison.

**Acceptance contract**

Only order changes; centre remains one.

**Failure mode**

Explain ringing/detail changes without calling sharper transitions universally better.

## 12. Demonstrate Ringing

**Purpose**

Reveal oscillation caused by sharp spectral truncation.

**Inputs**

Sharp-edge example and compared low-pass masks.

**Procedure**

1. Create or use a sharp-edged synthetic square image.
2. Apply abrupt Ideal low-pass and smoother alternatives.
3. Inspect overshoot/undershoot near edges.
4. Measure ringing with the same overshoot metric for each method.

**Produces**

Filtered edge profiles and overshoot.

**Acceptance contract**

Measure overshoot on raw reconstruction, before clipping or rescaling.

**Failure mode**

Do not hide negative/overshooting values in the metric computation.

## 13. Implement High-Pass Filters

**Purpose**

Remove the DC component with complementary filters.

**Inputs**

Low-pass masks and the same source.

**Procedure**

1. Create high-pass masks as `1 - low_pass_mask` for matching families/cutoffs.
2. Verify spectrum-center response is approximately zero.
3. Apply high-pass masks to the same source image.
4. Keep signed/high-frequency results for computation and normalize only display copies.

**Produces**

H_high=1-H_low and high-pass results.

**Acceptance contract**

Centre near zero; keep signed numerical outputs.

**Failure mode**

Do not cast signed detail to uint8 before measuring it.

## 14. Implement High-Boost Sharpening

**Purpose**

Add controlled high-frequency detail.

**Inputs**

Original image, high-pass result and gain.

**Procedure**

1. Use a selected high-pass result as the detail component.
2. Combine original image plus a controlled gain times the detail.
3. Clip only the final display/storage image to the valid intensity range.
4. Save original, detail, and sharpened comparison.

**Produces**

Sharpened image and comparison.

**Acceptance contract**

Same raw detail field; gain zero gives the original.

**Failure mode**

Investigate excessive noise/halos; clip only the displayed/stored version.

## 15. Validate the Convolution Theorem

**Purpose**

Compare equivalent linear convolution calculations.

**Inputs**

Image crop and kernel with matching alignment.

**Procedure**

1. Take the declared image crop and normalized spatial kernel.
2. Compute direct convolution with zero boundary padding and a flipped kernel.
3. Pad FFT dimensions to `image_shape + kernel_shape - 1` to obtain linear, not circular, convolution.
4. Multiply spectra, inverse-transform and crop using the same kernel-centre offset as the spatial result.
5. Assert equal shapes, finite FFT output and maximum absolute error below `1e-8`.

**Produces**

Spatial reference, FFT convolution and error.

**Acceptance contract**

Same zero padding, kernel centring and output crop; current maximum-error assertion is below 1e-8.

**Failure mode**

Resolve boundary/shift mismatch before loosening the tolerance.

## 16. Implement Band-Pass and Band-Reject Filters

**Purpose**

Select or reject a radial frequency interval.

**Inputs**

Shape and ordered lower/upper cutoffs.

**Procedure**

1. Define lower and upper radial cutoffs.
2. Set band-pass to retain only frequencies within the annulus.
3. Set band-reject as its complement.
4. Apply both to the same source and save the diagnostic.

**Produces**

Band-pass mask and complementary reject mask.

**Acceptance contract**

Masks sum to one and match the image dimensions.

**Failure mode**

Correct reversed cutoffs; retain the same source when comparing outputs.

## 17. Inspect Periodic Interference Spectra

**Purpose**

Locate candidate periodic interference.

**Inputs**

Corrupted images and centred spectra.

**Procedure**

1. Load periodic-interference examples.
2. Compute centered log-magnitude spectra.
3. Inspect symmetric off-center peaks associated with periodic noise.
4. Protect the DC/low-frequency center when later searching for peaks.

**Produces**

Log-magnitude interference overview.

**Acceptance contract**

Real-image spectra show conjugate structure; distinguish peaks from scene texture.

**Failure mode**

Treat peaks as candidates, not automatic proof of noise.

## 18. Detect Strong Spectral Peaks

**Purpose**

Find separated peaks away from DC.

**Inputs**

Centred spectrum; exclusion radius; separation; count.

**Procedure**

1. Create a peak-search helper that masks the central low-frequency region.
2. Rank candidate locations by spectrum magnitude.
3. Enforce minimum spacing between selected peaks.
4. Return peak coordinates in centered-spectrum indexing.

**Produces**

Selected (row,column) coordinates.

**Acceptance contract**

Review that candidates lie outside DC exclusion and inside bounds.

**Failure mode**

Current greedy routine needs manual review if too few eligible peaks exist.

## 19. Build Symmetric Notch-Reject Masks

**Purpose**

Suppress paired periodic components.

**Inputs**

Peak locations, image shape and notch radius.

**Procedure**

1. For each detected peak include its conjugate-symmetric counterpart about the center.
2. Set small neighborhoods around peak pairs to zero and all other mask values to one.
3. Multiply the centered spectrum by this mask and inverse transform.
4. Save spectrum, notch mask, and restored image.

**Produces**

Notch mask and restored image.

**Acceptance contract**

Check conjugate pairing and DC preservation for selected peaks; current helper reflects indices geometrically.

**Failure mode**

Boundary/Nyquist peaks need modular-index handling not implemented here; do not claim universal odd/even-shape coverage.

## 20. Remove Moiré Interference

**Purpose**

Apply targeted rejection to moire interference.

**Inputs**

Moire image and selected peak pattern.

**Procedure**

1. Load the moiré image and inspect its spectrum.
2. Locate the periodic peak pattern.
3. Construct notch rejection around those peaks instead of broad low-pass suppression.
4. Inverse transform and compare corrected versus corrupted image.

**Produces**

Corrected image and spectral diagnostic.

**Acceptance contract**

Removed energy corresponds to the inspected peaks; assess lost scene texture.

**Failure mode**

Retune placement/radius if noise persists or structure is removed.

## 21. Correct Low-Frequency Illumination Variation

**Purpose**

Estimate and compensate slowly varying illumination.

**Inputs**

Spot-shading image and low-pass scale.

**Procedure**

1. Load the spot-shading image as grayscale.
2. Estimate illumination using the Gaussian low-pass mask with cutoff 18.
3. Divide the image by `illumination + 1e-6`.
4. Normalize this corrected array to the unit interval for display and save the original/illumination/corrected comparison.

**Produces**

Illumination estimate and corrected display.

**Acceptance contract**

Inspect the actual correction expression and denominator protection in main.ipynb.

**Failure mode**

Report halos or unstable correction; display normalization alone is not radiometric validation.

## 22. Run Cutoff-Sensitivity Experiments

**Purpose**

Measure the effect of cutoff independently.

**Inputs**

One family, one source, multiple cutoffs.

**Procedure**

1. Choose one filter family and evaluate several declared cutoff values.
2. Keep all other parameters fixed.
3. Measure detail/reconstruction/ringing quantities where relevant.
4. Plot or tabulate the controlled sensitivity trend.

**Produces**

Controlled cutoff comparison.

**Acceptance contract**

Keep order and other settings fixed.

**Failure mode**

Do not compare different sources as a cutoff experiment.

## 23. Run Quantitative Checks

**Purpose**

Interpret each quantitative measure with its reference.

**Inputs**

Raw results and stated reference images.

**Procedure**

1. Compute MSE/PSNR when a meaningful reference exists.
2. Compute mean-gradient magnitude for detail preservation.
3. Compute ringing overshoot for sharp-edge experiments.
4. Verify numerical results are finite, except intentional infinite PSNR at zero error.

**Produces**

MSE, PSNR, gradient and overshoot measures.

**Acceptance contract**

Zero-error PSNR may be infinite; MSE versus input measures change, not restoration accuracy.

**Failure mode**

Do not label input-relative fidelity as ground-truth denoising quality.

## 24. Validate Frequency-Domain Invariants

**Purpose**

Check transform and mask invariants.

**Inputs**

Round-trip image and low/high-pass masks.

**Procedure**

1. Recheck FFT/IFFT round-trip accuracy.
2. Verify low-pass center response ≈1 and high-pass center response ≈0.
3. Verify masks and reconstructed images match source shape.
4. Reject non-finite arrays.

**Produces**

Numerical validation results.

**Acceptance contract**

Dtype-aware reconstruction check; LP DC=1 and HP DC=0; correct shapes and finite arrays.

**Failure mode**

Stop on failed invariants before interpreting filter performance.

## 25. Summarize Failure Signatures

**Purpose**

Connect observed artifacts to likely causes.

**Inputs**

Saved spectra, edge profiles and outputs.

**Procedure**

1. Record how excessive blur, ringing, residual periodic peaks, or bad shading correction appear.
2. Associate each signature with the responsible parameter/filter choice.
3. Base the summary only on diagnostics produced earlier.

**Produces**

Failure-signature discussion.

**Acceptance contract**

Claims refer to actual earlier diagnostics.

**Failure mode**

Describe an untested cause as a hypothesis rather than a measured result.

## 26. Compare Parameter Sensitivity

**Purpose**

Summarize controlled parameter experiments.

**Inputs**

Cutoff/order/notch/correction comparisons.

**Procedure**

1. Collect sensitivity results into one concise summary.
2. Compare changes caused by cutoff, Butterworth order, notch placement, or correction strength.
3. State which parameter controls which observed behaviour.

**Produces**

Parameter-sensitivity interpretation.

**Acceptance contract**

Name what was held fixed and what changed.

**Failure mode**

Avoid causal claims when several parameters changed together.

## 27. Build the Method-Selection Summary

**Purpose**

Select methods according to the image problem.

**Inputs**

Low/high/band/notch results.

**Procedure**

1. Compare Ideal, Gaussian, Butterworth, high-pass, band, and notch strategies by the problem they solve.
2. Base the summary on measured/visualized outputs.
3. Do not introduce a new method at this stage.

**Produces**

Method-selection summary.

**Acceptance contract**

Separate smoothing, sharpening and periodic-noise removal objectives.

**Failure mode**

State tradeoffs when one method removes both noise and detail.

## 28. Implement the Integrated Frequency Workflow

**Purpose**

Reuse low-pass filtering as a complete example.

**Inputs**

Image path, family, cutoff and order.

**Procedure**

1. Define `run_frequency_filtering_workflow(image_path, filter_family="butterworth", cutoff=30, order=2)`.
2. Load the image and choose one of the three low-pass builders: ideal, Gaussian or Butterworth; reject other family names.
3. Apply the generic spectral-filter helper and check shapes, finite values and mask range.
4. Save the integrated figure, then return a dictionary containing image name, shape, settings, mask range and MSE/PSNR versus the input.
5. Do not describe the returned value as image arrays or a persisted JSON file: those are not its current API.

**Produces**

Saved integrated figure and returned report dictionary.

**Acceptance contract**

Supported families: ideal, gaussian, butterworth; finite same-shaped result and mask in [0,1].

**Failure mode**

Existing helper rejects unknown families; returned report is not an exported JSON audit file.

## 29. Validate Required Output Files

**Purpose**

Verify the declared artifacts.

**Inputs**

REQUIRED_OUTPUTS and OUTPUT_DIR.

**Procedure**

1. Define the literal `REQUIRED_OUTPUTS` list.
2. Verify every filename exists in `outputs/figures/`.
3. Raise `FileNotFoundError` listing missing artifacts.
4. Print the final validation-pass message only after numerical and file checks succeed.

**Produces**

Final output-presence check.

**Acceptance contract**

All 22 literal filenames from main.ipynb exist.

**Failure mode**

Raise with missing filenames; a presence check does not prove files were freshly regenerated.